In [45]:
import numpy as np
import pandas as pd
import re

In [ ]:

# Caminho/aba

path = r"C:\Users\silma\Projetos\vigimed\eda\artigo\scoping review - tables.xlsx"  # ajuste se necessário
sheet = "Extraction"

In [ ]:


# Colunas
tech_cols = [
    "disproportionality_techniques",
    "data_mining_algorithms_techiniques",
    "other_techniques",
]
drug_col = "drug_class"

df = pd.read_excel(path, sheet_name=sheet, dtype=str)
df[drug_col] = df[drug_col].fillna("")
for c in tech_cols:
    df[c] = df[c].fillna("")

def split_clean(s: str):
    if not s:
        return []
    return [p.strip() for p in s.split(",") if p.strip()]

def extract_atc_codes(text):
    if pd.isna(text):
        return []
    return re.findall(r"[A-Z]\d{2}[A-Z0-9]{0,4}", text)

df["atc_list"] = df["drug_class"].apply(extract_atc_codes)
df["atc_class"] = df["atc_list"].apply(
    lambda lst: list(set(code[:3] for code in lst))
)

In [8]:
drug = df[["article_id", drug_col]].copy()
drug.head()

,article_id,drug_class
0,1,"J01AA08 (Minocycline), J01BA01(Chloramphenicol..."
1,2,"A10BA02 (Metformin), A10BK (SGLT-2 Inhibitors)"
2,3,"N05BA01 (Diazepam), N05BA04 (Oxazepam), N05BA0..."
3,4,D11AH05 (Dupilumab)
4,5,"N05 (Psycholeptics), N06 (Psychoanaleptics)"


In [44]:
drug.query('article_id == "107"')

,article_id,drug_class,atc_level_1
106,107,"""V08AB (Watersoluble","""V0"
106,107,nephrotropic,nep
106,107,"low osmolar X-ray contrast media)""",low


In [32]:
# -----------------------
# 1) ATC por artigo: 1, 2, 3+
drug = df[["article_id", drug_col]].copy()
drug[drug_col] = drug[drug_col].apply(split_clean)
drug = drug.explode(drug_col)
drug = drug[drug[drug_col] != ""]  # opcional: remover vazios

drug.head()

,article_id,drug_class
0,1,J01AA08 (Minocycline)
0,1,J01BA01(Chloramphenicol)
0,1,J01MA14 (Moxifloxacin )
1,2,A10BA02 (Metformin)
1,2,A10BK (SGLT-2 Inhibitors)


In [33]:
drug["atc_level_1"] = drug[drug_col].str[:3]
drug.head()

,article_id,drug_class,atc_level_1
0,1,J01AA08 (Minocycline),J01
0,1,J01BA01(Chloramphenicol),J01
0,1,J01MA14 (Moxifloxacin ),J01
1,2,A10BA02 (Metformin),A10
1,2,A10BK (SGLT-2 Inhibitors),A10


In [34]:
drug.drug_class.nunique()

308

In [35]:
druh_atc_level_1 = drug[['article_id', 'atc_level_1']].drop_duplicates() 
druh_atc_level_1.head()

,article_id,atc_level_1
0,1,J01
1,2,A10
2,3,N05
3,4,D11
4,5,N05


In [36]:
druh_atc_level_1.atc_level_1.nunique()

63

In [39]:
druh_atc_level_1['atc_level_1'].value_counts(dropna=False)

atc_level_1
L01    43
N05    17
L04    17
A10    13
J01    13
       ..
nep     1
Gly     1
low     1
B02     1
M03     1
Name: count, Length: 63, dtype: int64

In [40]:
druh_atc_level_1 = (
    drug.groupby("article_id")['atc_level_1']
    .nunique()
    .reset_index(name="n_atc")
)
druh_atc_level_1['n_atc'] = np.where(druh_atc_level_1['article_id'] ==82, 3, druh_atc_level_1['n_atc'])

In [41]:
drug.query('article_id == "102"')

,article_id,drug_class,atc_level_1
101,102,A08AA10 (Sibutramine),A08
101,102,A10BG02 (Rosiglitazone),A10
101,102,A10BH01 (Sitagliptin),A10
101,102,A10BK02 (Canagliflozin),A10
101,102,N02AC05 (Propoxyphene),N02


In [42]:
drug.query('atc_level_1 == "nep"')

,article_id,drug_class,atc_level_1
106,107,nephrotropic,nep


In [43]:
drug.query('article_id == "107"')

,article_id,drug_class,atc_level_1
106,107,"""V08AB (Watersoluble","""V0"
106,107,nephrotropic,nep
106,107,"low osmolar X-ray contrast media)""",low


In [23]:
atc_dist = (
    druh_atc_level_1.assign(
        atc_level_1=lambda x: x["n_atc"].clip(upper=3)  # 3 representa 3+
    )
    .groupby("atc_level_1")
    .size()
    .rename("artigos")
    .reset_index()
)
atc_dist

,atc_level_1,artigos
0,1,138
1,2,10
2,3,14


In [24]:
drug.query('article_id == "82"')

,article_id,drug_class,atc_level_1
81,82,000000 All drugs,000


In [ ]:

atc_dist = (
    drug_counts.assign(
        bucket=lambda x: x["n_atc"].clip(upper=3)  # 3 representa 3+
    )
    .groupby("bucket")
    .size()
    .rename("artigos")
    .reset_index()
)
# Salvar
atc_dist.to_csv("dist_artigos_por_atc.csv", index=False)
drug_counts.to_csv("atc_por_artigo.csv", index=False)

# -----------------------
# 2) Explodir técnicas (todas as colunas)
parts = []
for col in tech_cols:
    tmp = df[["article_id", col]].copy()
    tmp[col] = tmp[col].apply(split_clean)
    tmp = tmp.explode(col)
    tmp = tmp[tmp[col].notna() & (tmp[col] != "")]
    parts.append(tmp.rename(columns={col: "technique"}).assign(source_col=col))

exploded = pd.concat(parts, ignore_index=True)

# (opcional) remover "Not used"
exploded_clean = exploded[exploded["technique"].str.lower() != "not used"]

# -----------------------
# 2a) Contagem de técnicas (todas colunas juntas)
tech_counts = (
    exploded_clean.groupby("technique")
    .size()
    .sort_values(ascending=False)
    .rename("count")
    .reset_index()
)
tech_counts.to_csv("contagem_tecnicas.csv", index=False)

# -----------------------
# 3) Artigos com 1 técnica, 2 técnicas, 3+ (considerando todas as colunas)
tech_per_article = (
    exploded_clean.groupby("article_id")["technique"]
    .nunique()
    .reset_index(name="n_techniques")
)
tech_dist = (
    tech_per_article.assign(
        bucket=lambda x: x["n_techniques"].clip(upper=3)  # 3 representa 3+
    )
    .groupby("bucket")
    .size()
    .rename("artigos")
    .reset_index()
)
tech_dist.to_csv("dist_artigos_por_tecnicas.csv", index=False)
tech_per_article.to_csv("tecnicas_por_artigo.csv", index=False)

# -----------------------
# 4) Combinações de técnicas mais usadas (set por artigo)
combos = (
    exploded_clean.groupby("article_id")["technique"]
    .apply(lambda s: tuple(sorted(set(s))))  # conjunto ordenado para normalizar
    .reset_index(name="tech_combo")
)
combo_counts = (
    combos.groupby("tech_combo")
    .size()
    .sort_values(ascending=False)
    .rename("artigos")
    .reset_index()
)
combo_counts.to_csv("combos_tecnicas.csv", index=False)

print("Gerados:")
print("- dist_artigos_por_atc.csv (1/2/3+ ATC)")
print("- atc_por_artigo.csv")
print("- contagem_tecnicas.csv")
print("- dist_artigos_por_tecnicas.csv (1/2/3+ técnicas)")
print("- tecnicas_por_artigo.csv")
print("- combos_tecnicas.csv")

Gerados:
- dist_artigos_por_atc.csv (1/2/3+ ATC)
- atc_por_artigo.csv
- contagem_tecnicas.csv
- dist_artigos_por_tecnicas.csv (1/2/3+ técnicas)
- tecnicas_por_artigo.csv
- combos_tecnicas.csv


In [8]:
exploded_clean

,article_id,technique,source_col
0,1,Information Component,disproportionality_techniques
1,1,Proportional Reporting Ratio,disproportionality_techniques
2,1,Reporting Odds Ratio,disproportionality_techniques
3,2,Reporting Odds Ratio,disproportionality_techniques
4,3,Empirical Bayes Geometric Mean,disproportionality_techniques
...,...,...,...
968,161,Descriptive Statistics,other_techniques
969,162,Calinski–Harabasz Index,other_techniques
970,162,Davies–Bouldin Index,other_techniques
971,162,Dunn Index,other_techniques
